# Smoothness × recency — five paired seeds

Four continuations from an identical step-6,300 checkpoint per seed. Each question is presented exactly 45 times. The smooth/stepwise contrast matches Binding count, average presentation time, and first/last-window proportions within each recency condition. This notebook runs all five seeds directly and resumes completed runs.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEEDS = [42, 43, 44, 45, 46]
BATCH_SIZE, EVAL_EVERY = 50, 100
LR, WEIGHT_DECAY = 1e-4, 0.0
MODEL_DIM, PATCH_SIZE, MODEL_DEPTH = 48, 4, 1
AUGMENT, DEVICE = True, "cuda"
DATA_REL = Path("outputs/medium_world/dataset")
OUT_REL = Path("outputs/medium_world/experiments/smoothness_recency/five_seeds")
SOURCE_REL = Path("outputs/medium_world/experiments/transition_rehearsal/five_seeds")
DATA_DIR, OUT_DIR = PROJECT_ROOT / DATA_REL, PROJECT_ROOT / OUT_REL
SOURCE_DIR = PROJECT_ROOT / SOURCE_REL
RESULTS_FILE = OUT_DIR / "schedule_comparison.json"
TRACE_DIR, CHECKPOINT_DIR = OUT_DIR / "prediction_traces", OUT_DIR / "checkpoints"

In [ ]:
import copy, hashlib, json, random, shutil
from collections import Counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display
from src.medium_visual_world.learning import (
    TASKS, MediumDataset, AttributeBindingViT, augment_images, set_seed, environment
)
from src.medium_visual_world.smoothness_recency import (
    STRATEGIES, PROFILES, PREFIX_STEPS, TOTAL_STEPS, WINDOW_STEPS,
    make_smoothness_blocks, profile_stats
)
from src.medium_visual_world.transition_control import SOURCE
from src.medium_visual_world.prediction_tracking import (
    PredictionTrace, evaluate_predictions, load_trace
)

device = torch.device(DEVICE)
if device.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but not available; change DEVICE to 'cpu' if needed")
torch.set_num_threads(min(4, torch.get_num_threads()))
train_data = MediumDataset(DATA_DIR / "train.jsonl")
id_dev = MediumDataset(DATA_DIR / "id_dev.jsonl")
ood_dev = MediumDataset(DATA_DIR / "ood_dev.jsonl")
datasets = {"id_dev": id_dev, "ood_dev": ood_dev}
hashes = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
          for name in ("train.jsonl", "id_dev.jsonl", "ood_dev.jsonl")}
EXPECTED_HASHES = {
    "train.jsonl": "69deccacbac6db25f7ffc8165f01545483d856855282ea640997f6660ce27c26",
    "id_dev.jsonl": "ece4b831f1f7fe6efad42b18aca499f12c2d573baa76fc218864bb56fe24b11a",
    "ood_dev.jsonl": "208a7566cc2082fa4438ec0d003430d0927c54bf82c55bb40fa885ef4961e0aa",
}
assert hashes == EXPECTED_HASHES, "Dataset differs from the completed transition experiment"
print(device, "|", len(SEEDS), "seeds ×", len(STRATEGIES), "strategies | branch at", PREFIX_STEPS)

cuda | 5 seeds × 4 strategies | branch at 6300


## Verify frozen schedules

The 2×2 design uses nine 300-update windows. Within either recency level, smooth and stepwise have identical total per-task presentations, exact mean Binding exposure time, and matching first/last-window proportions. All schedules share an identical 6,300-step parent.

In [3]:
def verify_real_plan(seed):
    parent = None
    for strategy in STRATEGIES:
        prefix, suffix, plan = make_smoothness_blocks(train_data.rows, seed, strategy)
        assert parent is None or prefix == parent
        parent = prefix
        assert len(suffix) == 9 and all(len(ids) == 15000 for _, ids in suffix)
        assert set(Counter(i for _, ids in prefix+suffix for i in ids).values()) == {45}
        assert all(n == 112500 for n in plan["total_presentations_by_task"].values())
    return [{"seed": seed, "strategy": s, **profile_stats(PROFILES[s])} for s in STRATEGIES]

profile_table = pd.DataFrame(verify_real_plan(SEEDS[0]))
display(profile_table[["strategy", "pair_percentage_by_window", "mean_pair_optimizer_step",
                       "max_adjacent_percentage_point_jump", "squared_jump_score"]])
for seed in SEEDS:
    verify_real_plan(seed)
print("PASS: all five seeds have matched exposure, source prefixes, and frozen profiles")

,strategy,pair_percentage_by_window,mean_pair_optimizer_step,max_adjacent_percentage_point_jump,squared_jump_score
0,smooth_early,"[80, 70, 65, 60, 50, 45, 35, 25, 20]",7350.0,10,500
1,step_early,"[80, 74, 74, 46, 46, 46, 32, 32, 20]",7350.0,28,1160
2,smooth_late,"[20, 25, 35, 45, 50, 60, 65, 70, 80]",7950.0,10,500
3,step_late,"[20, 32, 32, 46, 46, 46, 74, 74, 80]",7950.0,28,1160


PASS: all five seeds have matched exposure, source prefixes, and frozen profiles


## Shared parent checkpoints

Each seed uses the exact `prereq_mixed_pair_focus` history to step 6,300. **If a verified checkpoint and prefix trace from `transition_rehearsal/five_seeds` are available, reuse them**; otherwise train and save a new parent. Each continuation restores both model and full AdamW/RNG state.

In [4]:
CONFIG = {"design": "smoothness_recency_v1", "source": SOURCE,
          "strategies": list(STRATEGIES), "profiles": {k: list(v) for k,v in PROFILES.items()},
          "seeds": SEEDS, "batch_size": BATCH_SIZE, "eval_every": EVAL_EVERY,
          "learning_rate": LR, "weight_decay": WEIGHT_DECAY,
          "model_dim": MODEL_DIM, "patch_size": PATCH_SIZE, "depth": MODEL_DEPTH,
          "augmentation": AUGMENT, "prefix_steps": PREFIX_STEPS,
          "total_steps": TOTAL_STEPS, "window_steps": WINDOW_STEPS, "trace_schema": 1}

def capture_rng():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def model_digest(model):
    digest = hashlib.sha256()
    for key, value in sorted(model.state_dict().items()):
        digest.update(key.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def new_model_optimizer():
    model = AttributeBindingViT(dim=MODEL_DIM, patch_size=PATCH_SIZE,
                                depth=MODEL_DEPTH).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    return model, optimizer


def new_trace(seed, schedule, parent_hash=None):
    return PredictionTrace(datasets, metadata={"seed": seed, "schedule": schedule,
                            "dataset_hashes": hashes, "parent_sha256": parent_hash})


def evaluate_at(model, optimizer, trace, history, step, phase, seen, train_loss=None):
    scores = {}
    for split, data in datasets.items():
        result, preds = evaluate_predictions(model, data, device, BATCH_SIZE)
        scores[split] = result
        trace.add(split, step, preds)
    history.append({"step": step, "phase": phase,
                    "lr": optimizer.param_groups[0]["lr"],
                    "batch_train_loss": train_loss,
                    "cumulative_task_presentations": dict(seen), **scores})
    print(f"{phase:16} step={step:4} ID={scores['id_dev']['by_template']['pair']:.3f} "
          f"OOD={scores['ood_dev']['by_template']['pair']:.3f}", flush=True)


def fit_blocks(model, optimizer, blocks, trace, history, start_step, seen):
    step = start_step
    transitions = []
    for phase, indices in blocks:
        phase_start = step
        accumulated, batches = 0.0, 0
        for start in range(0, len(indices), BATCH_SIZE):
            batch_ids = indices[start:start+BATCH_SIZE]
            x, q, y = train_data.batch(batch_ids, device)
            if AUGMENT:
                x = augment_images(x)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(x, q), y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss at step {step}")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            step += 1
            accumulated += float(loss.item())
            batches += 1
            seen.update(train_data.tasks[i] for i in batch_ids)
            if step % EVAL_EVERY == 0:
                evaluate_at(model, optimizer, trace, history, step, phase, seen,
                            accumulated/batches)
                accumulated, batches = 0.0, 0
        assert history[-1]["step"] == step
        transitions.append({"phase":phase,"start_step":phase_start,"end_step":step})
    return step, transitions


def prefix_path(seed):
    return CHECKPOINT_DIR / f"prefix_seed_{seed}.pt"


def prefix_trace_path(seed):
    return TRACE_DIR / f"prefix_seed_{seed}.npz"


def load_parent(seed):
    path = prefix_path(seed)
    if not path.exists() or not prefix_trace_path(seed).exists():
        return None
    # Local checkpoint created by this notebook; never load untrusted .pt files.
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    assert ckpt["seed"] == seed and ckpt["step"] == PREFIX_STEPS
    assert ckpt["config"] == CONFIG and ckpt["dataset_hashes"] == hashes
    t = load_trace(prefix_trace_path(seed))
    for split,data in datasets.items():
        block=t["splits"][split]
        assert np.array_equal(block["steps"],np.arange(0,PREFIX_STEPS+1,EVAL_EVERY))
        assert list(block["ids"]) == [str(r["id"]) for r in data.rows]
    assert len(ckpt["history"]) == PREFIX_STEPS//EVAL_EVERY+1
    return ckpt




def import_prior_parent(seed):
    source_ckpt = SOURCE_DIR / "checkpoints" / f"prefix_seed_{seed}.pt"
    source_trace = SOURCE_DIR / "prediction_traces" / f"prefix_seed_{seed}.npz"
    if not source_ckpt.exists() or not source_trace.exists():
        return None
    # Trusted local checkpoint from the previous experiment, not an online download.
    ckpt = torch.load(source_ckpt, map_location="cpu", weights_only=False)
    old = ckpt["config"]
    assert ckpt["seed"] == seed and ckpt["step"] == PREFIX_STEPS
    assert ckpt["dataset_hashes"] == hashes and old["source"] == SOURCE
    for key, value in (("learning_rate", LR), ("weight_decay", WEIGHT_DECAY),
                       ("model_dim", MODEL_DIM), ("patch_size", PATCH_SIZE),
                       ("depth", MODEL_DEPTH), ("augmentation", AUGMENT),
                       ("batch_size", BATCH_SIZE), ("eval_every", EVAL_EVERY)):
        assert old[key] == value, f"Prior parent settings differ: {key}"
    t = load_trace(source_trace)
    assert all(np.array_equal(t["splits"][s]["steps"],
                 np.arange(0,PREFIX_STEPS+1,EVAL_EVERY)) for s in datasets)
    for split, data in datasets.items():
        assert list(t["splits"][split]["ids"]) == [str(row["id"]) for row in data.rows]
    ckpt["parent_provenance"] = SOURCE_REL.as_posix()
    ckpt["config"] = CONFIG
    CHECKPOINT_DIR.mkdir(parents=True,exist_ok=True)
    TRACE_DIR.mkdir(parents=True,exist_ok=True)
    tmp = prefix_path(seed).with_suffix(".pt.tmp")
    torch.save(ckpt,tmp);tmp.replace(prefix_path(seed))
    shutil.copy2(source_trace,prefix_trace_path(seed))
    print(f"Imported verified step-{PREFIX_STEPS} parent for seed {seed}")
    return ckpt


def make_parent(seed):
    found = load_parent(seed)
    if found is not None:
        print(f"Reusing verified parent: seed {seed}")
        return found
    imported = import_prior_parent(seed)
    if imported is not None:
        return imported
    set_seed(seed)
    model, optimizer = new_model_optimizer()
    prefix, _, _ = make_smoothness_blocks(train_data.rows,seed,STRATEGIES[0])
    trace = new_trace(seed,"prefix")
    history,seen = [],Counter()
    evaluate_at(model,optimizer,trace,history,0,"initial",seen)
    step,transitions = fit_blocks(model,optimizer,prefix,trace,history,0,seen)
    assert step == PREFIX_STEPS
    parent_hash = model_digest(model)
    ckpt = {"seed":seed,"step":step,"config":CONFIG,"dataset_hashes":hashes,
            "model":model.state_dict(),"optimizer":optimizer.state_dict(),
            "rng":capture_rng(),"parent_sha256":parent_hash,
            "history":history,"seen":dict(seen),"transitions":transitions,
            "parent_provenance":"trained_here"}
    TRACE_DIR.mkdir(parents=True,exist_ok=True)
    CHECKPOINT_DIR.mkdir(parents=True,exist_ok=True)
    trace.save(prefix_trace_path(seed))
    tmp = prefix_path(seed).with_suffix(".pt.tmp")
    torch.save(ckpt,tmp);tmp.replace(prefix_path(seed))
    print(f"Trained and saved parent for seed {seed} | SHA {parent_hash[:12]}")
    return ckpt

## Run all 20 paired continuations

Run All trains the four suffix strategies for each seed 42–46. Each completed run saves its 91-checkpoint trace and JSON summary. Reruns verify and skip finished runs; interrupted runs are retried from their shared parent. No prior results are overwritten.

In [5]:
def trace_from_parent(seed, strategy, parent_hash):
    prefix = load_trace(prefix_trace_path(seed))
    trace = new_trace(seed, strategy, parent_hash)
    for split in datasets:
        block = prefix["splits"][split]
        for index, step in enumerate(block["steps"]):
            trace.add(split, int(step), {"predicted_label": block["predictions"][index],
                                         "probability_yes": block["probabilities"][index]})
    return trace


def trace_path(seed, strategy):
    return TRACE_DIR / f"{strategy}_seed_{seed}.npz"


def verify_trace(seed, strategy, parent_hash):
    trace = load_trace(trace_path(seed, strategy))
    assert trace["metadata"]["seed"] == seed and trace["metadata"]["schedule"] == strategy
    assert trace["metadata"]["parent_sha256"] == parent_hash
    assert trace["metadata"]["dataset_hashes"] == hashes
    for split, data in datasets.items():
        block = trace["splits"][split]
        assert list(block["ids"]) == [str(row["id"]) for row in data.rows]
        assert np.array_equal(block["steps"], np.arange(0,TOTAL_STEPS+1,EVAL_EVERY))
        assert block["predictions"].shape == (91,len(data))
        assert np.isfinite(block["probabilities"]).all()
    return trace


def binding_auc(history, begin=0, end=TOTAL_STEPS, split="id_dev"):
    subset = [row for row in history if begin <= row["step"] <= end]
    assert subset[0]["step"] == begin and subset[-1]["step"] == end
    return sum((b["step"]-a["step"])*(a[split]["by_template"]["pair"]+b[split]["by_template"]["pair"])/2
               for a,b in zip(subset[:-1],subset[1:]))/(end-begin)


def continue_one(seed, strategy, parent):
    assert parent["step"] == PREFIX_STEPS
    prefix, suffix, plan = make_smoothness_blocks(train_data.rows, seed, strategy)
    # Build model/optimizer *before* restoring saved RNG states.
    model, optimizer = new_model_optimizer()
    model.load_state_dict(parent["model"])
    optimizer.load_state_dict(copy.deepcopy(parent["optimizer"]))
    assert model_digest(model) == parent["parent_sha256"]
    restore_rng(parent["rng"])
    trace = trace_from_parent(seed, strategy, parent["parent_sha256"])
    history, seen = copy.deepcopy(parent["history"]), Counter(parent["seen"])
    step, transitions = fit_blocks(model, optimizer, suffix, trace, history,
                                   PREFIX_STEPS, seen)
    assert step == TOTAL_STEPS and len(history) == 91
    assert all(seen[task] == 112500 for task in TASKS)
    trace.save(trace_path(seed, strategy))
    verify_trace(seed, strategy, parent["parent_sha256"])
    return {"seed": seed, "schedule": strategy, "status":"completed",
            "steps_run": step, "parent_seed": seed,
            "parent_sha256": parent["parent_sha256"],
            "phase_plan": plan,
            "transitions": parent["transitions"] + transitions,
            "history": history, "final": history[-1],
            "id_pair_aulc": binding_auc(history),
            "id_pair_suffix_aulc": binding_auc(history, PREFIX_STEPS),
            "ood_pair_suffix_aulc": binding_auc(history, PREFIX_STEPS, split="ood_dev"),
            "prediction_trace": trace_path(seed,strategy).relative_to(OUT_DIR).as_posix()}


def write_report(report):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    temp = RESULTS_FILE.with_suffix(".json.tmp")
    temp.write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")
    temp.replace(RESULTS_FILE)

if RESULTS_FILE.exists():
    report = json.loads(RESULTS_FILE.read_text(encoding="utf-8"))
    if report["config"] != CONFIG or report["dataset_hashes"] != hashes:
        raise ValueError("Existing results use different configuration/dataset hashes")
else:
    report = {"status": "exploratory", "config": CONFIG, "dataset_hashes": hashes,
              "environment": environment(), "runs": []}

existing = {(r["seed"],r["schedule"]):r for r in report["runs"]}
assert len(existing) == len(report["runs"]), "Duplicate completed result entries"
for seed in SEEDS:
    parent = make_parent(seed)
    for strategy in STRATEGIES:
        key = (seed,strategy)
        if key in existing:
            assert existing[key]["status"] == "completed"
            assert existing[key]["parent_sha256"] == parent["parent_sha256"]
            verify_trace(seed,strategy,parent["parent_sha256"])
            print("Skipping verified run:",seed,strategy)
            continue
        result = continue_one(seed,strategy,parent)
        report["runs"].append(result)
        existing[key] = result
        write_report(report)
        print("Saved",seed,strategy,"→",RESULTS_FILE)
assert len(existing) == len(SEEDS)*len(STRATEGIES)
print("PASS:",len(existing),"completed runs, 5 shared parent checkpoints")

Imported verified step-6300 parent for seed 42
window_1         step=6400 ID=0.609 OOD=0.635
window_1         step=6500 ID=0.648 OOD=0.627
window_1         step=6600 ID=0.660 OOD=0.667
window_2         step=6700 ID=0.652 OOD=0.657
window_2         step=6800 ID=0.676 OOD=0.640
window_2         step=6900 ID=0.668 OOD=0.650
window_3         step=7000 ID=0.680 OOD=0.672
window_3         step=7100 ID=0.660 OOD=0.660
window_3         step=7200 ID=0.676 OOD=0.652
window_4         step=7300 ID=0.672 OOD=0.652
window_4         step=7400 ID=0.676 OOD=0.675
window_4         step=7500 ID=0.676 OOD=0.662
window_5         step=7600 ID=0.680 OOD=0.660
window_5         step=7700 ID=0.664 OOD=0.647
window_5         step=7800 ID=0.680 OOD=0.685
window_6         step=7900 ID=0.672 OOD=0.655
window_6         step=8000 ID=0.660 OOD=0.667
window_6         step=8100 ID=0.676 OOD=0.632
window_7         step=8200 ID=0.664 OOD=0.625
window_7         step=8300 ID=0.656 OOD=0.647
window_7         step=8400 ID=0.6

## Integrity summary

All 20 runs must have identical per-seed parents, full prediction traces, and exactly 112,500 presentations per task. Avoid committing `checkpoints/` files.

In [6]:
rows=[]
for run in report["runs"]:
    trace=verify_trace(run["seed"],run["schedule"],run["parent_sha256"])
    assert run["steps_run"] == TOTAL_STEPS and len(run["history"]) == 91
    assert all(run["final"]["cumulative_task_presentations"].get(t,0)==112500 for t in TASKS)
    rows.append({"seed":run["seed"],"strategy":run["schedule"],
                 "parent":run["parent_sha256"][:12],"steps":run["steps_run"],
                 "ID_binding":run["final"]["id_dev"]["by_template"]["pair"],
                 "OOD_binding":run["final"]["ood_dev"]["by_template"]["pair"]})
display(pd.DataFrame(rows))
assert len(rows)==20
print("PASS: 20 complete continuations, five shared parents, full traces")

,seed,strategy,parent,steps,ID_binding,OOD_binding
0,42,smooth_early,d5608a38fc99,9000,0.687500,0.6300
1,42,step_early,d5608a38fc99,9000,0.695312,0.6525
2,42,smooth_late,d5608a38fc99,9000,0.695312,0.6450
3,42,step_late,d5608a38fc99,9000,0.675781,0.6500
4,43,smooth_early,1bc72a828744,9000,0.644531,0.5600
5,43,step_early,1bc72a828744,9000,0.640625,0.5625
6,43,smooth_late,1bc72a828744,9000,0.656250,0.6150
7,43,step_late,1bc72a828744,9000,0.664062,0.6200
8,44,smooth_early,b1b91b420b42,9000,0.628906,0.5675
9,44,step_early,b1b91b420b42,9000,0.613281,0.5775


PASS: 20 complete continuations, five shared parents, full traces
